In [ ]:
## 🚀 OPTIONAL EXPERIMENT SCRIPT (APPENDIX D)
## Structure: CONFIG -> Setup -> Gen Data -> Train -> Zip
## Run directly on Kaggle Notebook Cell

import os
import shutil
import sys

# ==================================================================================
# 1. EXPERIMENT CONFIGURATION (YOU ONLY NEED TO EDIT HERE)
# ==================================================================================
CONFIG = {
    # --- IMPORTANT PARAMETERS (Change according to Appendix D) ---
    "train_size": 1000,             # Number of samples: 1000, 10000, or 100000
    "format": "10e-based",          # Format: "10e-based", "decimal", "character", "underscore", "words"
    "epochs": 200,                  # Recommended Epochs: 1k->200ep, 10k->100ep, 100k->20ep
    
    # --- DEFAULT CONFIGURATION (Usually keep as is) ---
    "digits": 15,                   # 15 digits addition (Appendix D) (Previously said 30 but typically 15 for D)
    "model_name": "t5-small",       # Model: t5-small or t5-base
    "batch_size": 128,               # Physical batch size (Reduce to 8 if OOM)
    "lr": 3e-4,                     # Learning rate
    "seed": 42,                     # Fixed seed for reproducibility
    "output_root": "outputs/custom_run", # Output directory
    "check_val_every_n_epoch": 10
}

# Automatically generate identifier for this run
RUN_ID = f"{CONFIG['format']}_{CONFIG['train_size']}samples_{CONFIG['epochs']}ep"
DATA_DIR = f"data/custom/{RUN_ID}"
OUTPUT_DIR = f"{CONFIG['output_root']}/{RUN_ID}"

# ==================================================================================
# HELPER FUNCTIONS
# ==================================================================================
def run_command(command):
    """Run shell command and print command for debugging"""
    print(f"\n⚡ [RUNNING]: {command}")
    exit_code = os.system(command)
    if exit_code != 0:
        print(f"❌ ERROR: Command failed with code {exit_code}")
        sys.exit(1) # Stop immediately if error
    return True

print("=" * 80)
print(f"🚀 STARTING EXPERIMENT: {RUN_ID}")
print(f"   - Task: Addition {CONFIG['digits']} digits")
print(f"   - Size: {CONFIG['train_size']} | Format: {CONFIG['format']}")
print(f"   - Epochs: {CONFIG['epochs']} | Batch: {CONFIG['batch_size']}")
print("=" * 80)

# ==================================================================================
# 2. SETUP ENVIRONMENT & INSTALLATION
# ==================================================================================
print("\n[1/5] 🛠️ Setting up environment...")

# Copy code from /kaggle/input if not present in /kaggle/working
if not os.path.exists("generate_data.py"):
    print("   -> Copying scripts from /kaggle/input...")
    # Try copying from subdirectories or root
    os.system("cp -r /kaggle/input/*/*.py . 2>/dev/null || cp /kaggle/input/*.py .")

# Install required libraries
run_command("pip install -q pytorch-lightning transformers num2words")
run_command(f"mkdir -p {DATA_DIR} {OUTPUT_DIR}")
print("✅ Environment ready.")

# ==================================================================================
# 3. GENERATE DATA
# ==================================================================================
print("\n[2/5] 📊 Generating Datasets...")

# Only regenerate if train file missing
if not os.path.exists(f"{DATA_DIR}/train.json"):
    # Train set: According to config size
    cmd_train = (
        f"python generate_data.py --mode single --count {CONFIG['train_size']} "
        f"--digits {CONFIG['digits']} --strategy balanced "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/train.json --seed {CONFIG['seed']}"
    )
    
    # Val set: Fixed 1000 samples
    cmd_val = (
        f"python generate_data.py --mode single --count 1000 "
        f"--digits {CONFIG['digits']} --strategy random "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/val.json --seed {CONFIG['seed'] + 1}"
    )
    
    # Test set: Fixed 2000 samples
    cmd_test = (
        f"python generate_data.py --mode single --count 2000 "
        f"--digits {CONFIG['digits']} --strategy random "
        f"--format {CONFIG['format']} --only_addition "
        f"--output {DATA_DIR}/test.json --seed {CONFIG['seed'] + 2}"
    )
    
    run_command(cmd_train)
    run_command(cmd_val)
    run_command(cmd_test)
    print("✅ Data generation completed.")
else:
    print("   -> Data files already exist. Skipping generation.")

# ==================================================================================
# 4. TRAINING MODEL
# ==================================================================================
print(f"\n[3/5] 🏋️ Training Model ({CONFIG['epochs']} epochs)...")

# Handle 'words' format requiring longer max_length
max_len = 512 if CONFIG['format'] == "words" or CONFIG['format'] == "10e-based" else 256

train_cmd = (
    f"python train.py "
    f"--train_file {DATA_DIR}/train.json "
    f"--val_file {DATA_DIR}/val.json "
    f"--test_file {DATA_DIR}/test.json "
    f"--output_dir {OUTPUT_DIR} "
    f"--model_name_or_path {CONFIG['model_name']} "
    f"--epochs {CONFIG['epochs']} "
    f"--train_batch_size {CONFIG['batch_size']} "
    f"--val_batch_size {CONFIG['batch_size']} "
    f"--max_seq_length {max_len} "
    f"--lr {CONFIG['lr']} "
    f"--seed {CONFIG['seed']} "
    f"--check_val_every_n_epoch {CONFIG['check_val_every_n_epoch']} "
    f"--accelerator gpu --devices 1 "
)

run_command(train_cmd)
print("✅ Training completed.")

# ==================================================================================
# 5. CHECK RESULTS & ZIP FILES
# ==================================================================================
print("\n[4/5] 📈 Quick Check Results...")
res_file = f"{OUTPUT_DIR}/results.json"
if os.path.exists(res_file):
    with open(res_file, 'r') as f:
        print(f"   -> Content of results.json: \n{f.read()}")
else:
    print("⚠️ Warning: results.json not found. Training might have failed/stopped early.")

print("\n[5/5] 📦 Zipping Results...")
zip_name = f"result_{RUN_ID}.zip"
run_command(f"zip -r {zip_name} {OUTPUT_DIR}")

print("\n" + "="*80)
print(f"🎉 DONE! Please download file '{zip_name}'.")
print("="*80)